# Regressione logistica

In [ ]:
import pandas as pd
import numpy as np


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler

from joblib import Parallel, delayed

from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [ ]:
def fit_single_fold(train_idx, test_idx, features, target, groups, C, penalty, solver, max_iter, fit_intercept, tol):
    # ========== DEBUGGING: Stampo indici train/test  ==========

    """print("?"*50 + "\nDebug\n" + "?"*50)
    print(f"\nFold {fold} - File: {csv_name}")
    print(f"  Train indice: {train_index[:10]})")
    print(f"  Test indice: {test_index[:10]})")
    print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
    print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
    print("?"*100)"""
    # ==========================================================

    X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
    y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]


    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # CORRETTO: parametri giusti per LogisticRegression
    lr = LogisticRegression(
        C=C,                        # Inverso della regolarizzazione
        penalty=penalty,            # Tipo di regolarizzazione ('l2', 'l1', etc.)
        solver=solver,              # Algoritmo di ottimizzazione
        max_iter=max_iter,          # Numero massimo iterazioni
        fit_intercept=fit_intercept, # Se includere bias
        tol=tol,                    # Tolleranza convergenza
        random_state=42
    )

    rf = MultiOutputClassifier(lr)
    rf.fit(X_train_scaled, y_train)
    y_pred = rf.predict(X_test_scaled)
    
    # DEBUG
    #score = f1_score(y_test, y_pred, average="micro")
    #print(f"Fold {fold} - {C=}, {penalty=}, {solver=}, Score={score:.4f}")
    
    return f1_score(y_test, y_pred, average="micro")


# Training

In [ ]:
def training(file_path, csv_name):
    df = pd.read_csv(file_path)
    
    # Mi definisco la lista dei target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Filtro solo le pazienti con PR valido
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro    
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)  
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """ 
        Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempio a Nan se è rimasto vuoto
    features = features.fillna(features.mean())

    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # Definisco gli iperparametri CORRETTI per LogisticRegression
    iperparametri = {
        'C': [0.01],                                # Inverso della regolarizzazione
        'penalty': ['l2'],                          # Tipo di regolarizzazione (l2 è compatibile con tutti i solver)
        'solver': ['saga'],                         # Algoritmo di ottimizzazione
        'max_iter': [100],                          # Numero massimo di iterazioni
        'fit_intercept': [True, False],             # Se includere il bias/intercetta
        'tol': [1e-4],                              # Tolleranza per il criterio di arresto
    } 

    for C in iperparametri['C']:
        for penalty in iperparametri['penalty']:
            for solver in iperparametri['solver']:
                for max_iter in iperparametri['max_iter']:
                    for fit_intercept in iperparametri['fit_intercept']:
                        for tol in iperparametri['tol']:
                            fold_scores = Parallel(n_jobs=-1)(
                                delayed(fit_single_fold)(train_idx, test_idx, features, target, groups, C, penalty, solver, max_iter, fit_intercept, tol) for train_idx, test_idx in cv.split(features, target, groups))

                            # calcolo media e deviazione standard degli score su tutte le fold
                            mean_score = np.mean(fold_scores)
                            std_score = np.std(fold_scores)
                            
                            scores.append({
                                'mean_score': mean_score,
                                'std_score': std_score,
                                'fold_scores': fold_scores,
                                'C': C,
                                'penalty': penalty,
                                'solver': solver,
                                'max_iter': max_iter,
                                'fit_intercept': fit_intercept,
                                'tol': tol
                            })

    return scores


# Lettura dei file

In [ ]:
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati per ogni dataset
for name, metrics_list in results_per_dataset.items():

    # Stampo solo i migliori
    best_result = max(metrics_list, key=lambda x: x['mean_score'])
    print(f"\nDataset: {name}:")
    print(f"  C: {best_result['C']}")
    print(f"  penalty: {best_result['penalty']}")
    print(f"  solver: {best_result['solver']}")
    print(f"  max_iter: {best_result['max_iter']}")
    print(f"  fit_intercept: {best_result['fit_intercept']}")
    print(f"  tol: {best_result['tol']}")
    print(f"  Mean F1-score: {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")


# Lo eseguo su colab, ecco l'output



Dataset: t2_medsam:
  C: 0.01
  penalty: l2
  solver: saga
  max_iter: 100
  fit_intercept: True
  tol: 0.0001
  Mean F1-score: 0.738 ± 0.071


Dataset: t2_preprocessed:
  C: 0.01
  penalty: l2
  solver: saga
  max_iter: 100
  fit_intercept: True
  tol: 0.0001
  Mean F1-score: 0.774 ± 0.070


Dataset: t2_original:
  C: 0.01
  penalty: l2
  solver: saga
  max_iter: 100
  fit_intercept: True
  tol: 0.0001
  Mean F1-score: 0.773 ± 0.058


Dataset: medsam_dynamic:
  C: 0.01
  penalty: l2
  solver: saga
  max_iter: 100
  fit_intercept: True
  tol: 0.0001
  Mean F1-score: 0.750 ± 0.059


Dataset: preprocessed_dynamic:
  C: 0.01
  penalty: l2
  solver: saga
  max_iter: 100
  fit_intercept: True
  tol: 0.0001
  Mean F1-score: 0.754 ± 0.052


Dataset: original_dynamic:
  C: 0.01
  penalty: l2
  solver: saga
  max_iter: 100
  fit_intercept: True
  tol: 0.0001
  Mean F1-score: 0.747 ± 0.050
